# Project 1: Exploratory Data Analysis (EDA) on Titanic Dataset
### Data Science & AI Internship
**Domain**: Data Science & Artificial Intelligence  
**Tools**: Python, Pandas, NumPy, Matplotlib, Seaborn  
**Dataset**: Titanic: Machine Learning from Disaster  

---
## 1. Project Objective
The goal of this project is to conduct a comprehensive exploratory data analysis (EDA) on the Titanic passenger dataset. Through systematic data inspection, robust handling of missing values, duplicate verification, group aggregations, and high-impact visual analytics, we aim to uncover the underlying socio-demographic factors that governed passenger survival during the 1912 maritime tragedy.

## 2. Import Libraries and Configure Environment

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Visual configuration
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["figure.autolayout"] = True
print("Libraries imported successfully!")

## 3. Requirement 1: Load Dataset from a Reliable Public Source
We fetch the official dataset directly from the Data Science Dojo repository, with an automatic fallback to our local CSV copy.

In [ ]:
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
local_path = "titanic.csv"

if os.path.exists(local_path):
    df = pd.read_csv(local_path)
    print(f"Loaded dataset from local file: {local_path}")
else:
    df = pd.read_csv(url)
    df.to_csv(local_path, index=False)
    print(f"Fetched dataset from public URL and saved to {local_path}")

print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")

## 4. Requirements 2 to 5: Inspect Dataset Dimensions, Sample Rows, Info & Statistics

In [ ]:
# Requirement 2: Display the first 5 rows
df.head()

In [ ]:
# Requirement 3: Dataset information (data types, non-null counts, memory usage)
df.info()

In [ ]:
# Requirement 4 & 5: Descriptive statistics for numerical features and row/column counts
print(f"Total Rows: {df.shape[0]}, Total Columns: {df.shape[1]}\n")
df.describe().T

In [ ]:
# Summary statistics for categorical attributes
cat_cols = df.select_dtypes(include=["object", "string", "category"]).columns
df[cat_cols].describe().T

## 5. Requirement 8: Identify and Handle Duplicate Records
Duplicate records can cause model overfitting and inflate frequency counts.

In [ ]:
duplicates = df.duplicated().sum()
print(f"Number of exact duplicate rows: {duplicates}")
if duplicates > 0:
    df = df.drop_duplicates()
    print(f"Removed duplicate rows. Updated row count: {len(df)}")
else:
    print("Verified: Every record in this dataset is distinct (0 duplicates).")

## 6. Requirements 6 & 7: Identify and Handle Missing Values Appropriately

### Methodology & Justifications:
1. **`Age` (Numerical - 177 missing / 19.87%)**:
   - **Method**: Median Imputation (`Age = 28.0`)
   - **Justification**: Age has positive skewness and extreme outliers (babies < 1 year to elderly seniors up to 80). Mean is pulled by extreme values, whereas median preserves the robust central tendency.
2. **`Embarked` (Categorical - 2 missing / 0.22%)**:
   - **Method**: Mode Imputation (`Embarked = 'S' / Southampton`)
   - **Justification**: 644 out of 889 known passengers (72.4%) embarked at Southampton. Imputing 2 missing records with the dominant category introduces negligible variance.
3. **`Cabin` (Categorical - 687 missing / 77.10%)**:
   - **Method**: Imputation with `'Unknown'`
   - **Justification**: Dropping 77% of rows would eliminate over 3/4 of the dataset. Dropping the column ignores that cabin allocation was heavily tied to deck location and passenger class. Imputing `'Unknown'` retains complete data fidelity.

In [ ]:
# Missing values check
missing_series = df.isnull().sum()
missing_df = pd.DataFrame({
    "Missing Count": missing_series,
    "Percentage (%)": ((missing_series / len(df)) * 100).round(2)
})
missing_df[missing_df["Missing Count"] > 0]

In [ ]:
# Create cleaned dataframe copy
df_clean = df.copy()

# 1. Age: Impute with median
median_age = df_clean["Age"].median()
df_clean["Age"] = df_clean["Age"].fillna(median_age)

# 2. Embarked: Impute with mode
mode_embarked = df_clean["Embarked"].mode()[0]
df_clean["Embarked"] = df_clean["Embarked"].fillna(mode_embarked)

# 3. Cabin: Impute with 'Unknown'
df_clean["Cabin"] = df_clean["Cabin"].fillna("Unknown")

print("Missing values after handling:")
print(df_clean.isnull().sum())
print(f"\nTotal remaining null values: {df_clean.isnull().sum().sum()}")

## 7. Requirement 9: Grouping and Aggregations
We examine survival rates across demographics and passenger characteristics.

In [ ]:
# 1. Survival rate by Gender
gender_agg = df_clean.groupby("Sex")["Survived"].agg(
    Passenger_Count="count",
    Survivors="sum",
    Survival_Rate="mean"
)
gender_agg["Survival_Rate (%)"] = (gender_agg["Survival_Rate"] * 100).round(2)
gender_agg

In [ ]:
# 2. Survival rate by Passenger Class
pclass_agg = df_clean.groupby("Pclass")["Survived"].agg(
    Passenger_Count="count",
    Survivors="sum",
    Survival_Rate="mean"
)
pclass_agg["Survival_Rate (%)"] = (pclass_agg["Survival_Rate"] * 100).round(2)
pclass_agg

In [ ]:
# 3. Average age by Passenger Class
df_clean.groupby("Pclass")["Age"].agg([
    ("Mean Age", "mean"),
    ("Median Age", "median"),
    ("Min Age", "min"),
    ("Max Age", "max")
]).round(2)

In [ ]:
# 4. Passenger counts and survival by Embarkation Port
port_map = {"S": "Southampton", "C": "Cherbourg", "Q": "Queenstown"}
embarked_agg = df_clean.groupby("Embarked")["Survived"].agg(
    Passenger_Count="count",
    Survival_Rate="mean"
)
embarked_agg["Port_Name"] = embarked_agg.index.map(port_map)
embarked_agg["Share (%)"] = ((embarked_agg["Passenger_Count"] / len(df_clean)) * 100).round(2)
embarked_agg["Survival_Rate (%)"] = (embarked_agg["Survival_Rate"] * 100).round(2)
embarked_agg[["Port_Name", "Passenger_Count", "Share (%)", "Survival_Rate (%)"]]

## 8. Requirements 10 & 13: Data Visualizations (Saved in `outputs/`)

In [ ]:
os.makedirs("outputs", exist_ok=True)

# Plot 1: Survival Count Plot
plt.figure(figsize=(7, 5))
ax = sns.countplot(data=df_clean, x="Survived", hue="Survived", palette=["#e74c3c", "#2ecc71"], legend=False)
plt.title("Titanic Passenger Survival Count", fontsize=14, fontweight="bold", pad=12)
plt.xticks([0, 1], ["Perished (0)", "Survived (1)"])
plt.xlabel("Outcome")
plt.ylabel("Passengers")
for p in ax.patches:
    val = int(p.get_height())
    ax.annotate(f"{val}\n({val/len(df_clean)*100:.1f}%)", (p.get_x() + p.get_width() / 2., val / 2), ha="center", va="center", color="white", fontweight="bold")
plt.savefig("outputs/01_survival_count.png", dpi=300)
plt.show()

In [ ]:
# Plot 2: Age Distribution Histogram
plt.figure(figsize=(8, 5))
sns.histplot(df_clean["Age"], kde=True, bins=30, color="#3498db", edgecolor="black", alpha=0.6)
plt.axvline(median_age, color="#e74c3c", linestyle="--", linewidth=2, label=f"Median: {median_age:.1f}")
plt.axvline(df_clean["Age"].mean(), color="#f39c12", linestyle="-", linewidth=2, label=f"Mean: {df_clean['Age'].mean():.1f}")
plt.title("Passenger Age Distribution", fontsize=14, fontweight="bold", pad=12)
plt.xlabel("Age (Years)")
plt.ylabel("Count")
plt.legend()
plt.savefig("outputs/02_age_distribution.png", dpi=300)
plt.show()

In [ ]:
# Plot 3: Survival Rate by Passenger Class
plt.figure(figsize=(7, 5))
ax = sns.barplot(data=pclass_agg.reset_index(), x="Pclass", y="Survival_Rate", hue="Pclass", palette="Blues_r", legend=False)
plt.title("Survival Rate by Passenger Class", fontsize=14, fontweight="bold", pad=12)
plt.xticks([0, 1, 2], ["1st Class", "2nd Class", "3rd Class"])
plt.ylabel("Survival Rate")
plt.ylim(0, 1.0)
for p in ax.patches:
    ax.annotate(f"{p.get_height()*100:.2f}%", (p.get_x() + p.get_width() / 2., p.get_height() + 0.02), ha="center", va="bottom", fontweight="bold")
plt.savefig("outputs/03_survival_by_pclass.png", dpi=300)
plt.show()

In [ ]:
# Plot 4: Survival Rate by Gender
plt.figure(figsize=(7, 5))
ax = sns.barplot(data=gender_agg.reset_index(), x="Sex", y="Survival_Rate", hue="Sex", palette=["#e84393", "#0984e3"], legend=False)
plt.title("Survival Rate by Gender", fontsize=14, fontweight="bold", pad=12)
plt.xticks([0, 1], ["Female", "Male"])
plt.ylabel("Survival Rate")
plt.ylim(0, 1.0)
for p in ax.patches:
    ax.annotate(f"{p.get_height()*100:.2f}%", (p.get_x() + p.get_width() / 2., p.get_height() + 0.02), ha="center", va="bottom", fontweight="bold")
plt.savefig("outputs/04_survival_by_gender.png", dpi=300)
plt.show()

In [ ]:
# Plot 5: Correlation Heatmap
plt.figure(figsize=(8, 6))
numeric_features = ["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare"]
sns.heatmap(df_clean[numeric_features].corr(), annot=True, cmap="coolwarm", fmt=".2f", square=True)
plt.title("Correlation Heatmap", fontsize=14, fontweight="bold", pad=12)
plt.savefig("outputs/05_correlation_heatmap.png", dpi=300)
plt.show()

## 9. Requirement 11: Five Useful Statistical Findings

1. **Overall Low Survival Rate (38.38%)**:
   - 342 passengers survived out of 891, while 549 (61.62%) perished. This reflects the catastrophic shortage of lifeboats.
2. **Gender Asymmetry in Survival (74.20% Female vs. 18.89% Male)**:
   - Females were ~3.9 times more likely to survive than males. This validates the strict adherence to the 'women and children first' evacuation protocol.
3. **Socioeconomic Stratification (62.96% 1st Class vs. 24.24% 3rd Class)**:
   - First-class passengers enjoyed a survival rate 2.6 times higher than third-class passengers, driven by proximity to upper decks and preferential lifeboat loading.
4. **Class-Age Gradient**:
   - The average age decreases steadily with passenger class: 1st class (36.81 years), 2nd class (29.77 years), 3rd class (25.93 years). Higher classes contained established, older patrons, while 3rd class consisted mostly of youthful emigrants.
5. **Cherbourg Boarding Advantage (55.36% Survival)**:
   - Passengers embarking at Cherbourg had the highest survival rate (55.36% vs. 33.90% at Southampton), because over 50.6% of Cherbourg boarders held first-class tickets.

## 10. Requirement 12: Export Cleaned Dataset

In [ ]:
df_clean.to_csv("cleaned_titanic.csv", index=False)
print(f"Successfully saved clean dataset to cleaned_titanic.csv with {df_clean.shape[0]} rows and {df_clean.shape[1]} columns.")